# TREC-DL pairwise attack flips

This notebook records pairwise preference reversals from `LLM_prompt_attack/outputs`. A **flipped pair** is a valid pair instance for which the attacked model label differs from its clean label. Because the experiment includes both document orders, the flipped-pair count is not a count of unique passages.

For TREC-DL 2019, completed Qwen3-32B back-position runs cover DOH and DCH with standard and defense prompts. GPT-OSS-20B also has a standard DOH result. For TREC-DL 2020 DOH, Qwen3-32B standard flipped **3,773/4,096 pairs (92.11%)**; Llama 3 8B standard flipped **2,462/4,096 (60.11%)** and defense flipped **2,318/4,096 (56.59%)**.

## Reproduction table

The next cell regenerates the grouped comparison table from completed JSONL files. Each attack group has `Orig`, `Ours`, and `Defense` columns; `Ours` contains standard-prompt ASR and `Defense` contains defense-prompt ASR. Unavailable cells remain blank. The standalone outputs are [`attack_table.html`](attack_table.html), [`attack_table.md`](attack_table.md), and [`attack_table.csv`](attack_table.csv).

In [5]:
import runpy
from pathlib import Path
from IPython.display import HTML

root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "LLM_prompt_attack").is_dir()
)
runpy.run_path(str(root / "Results" / "update_attack_table.py"), run_name="__main__")
HTML((root / "Results" / "attack_table.html").read_text(encoding="utf-8"))

Updated attack table with 12 recorded result(s).


In [6]:
import json
from pathlib import Path


def find_repository_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "LLM_prompt_attack").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


root = find_repository_root()
output_dir = root / "LLM_prompt_attack" / "outputs"
summaries = []

for result_path in sorted(output_dir.rglob("*.jsonl")):
    with result_path.open(encoding="utf-8") as result_file:
        for line in result_file:
            record = json.loads(line)
            if record.get("ranking_scheme") != "pairwise":
                continue
            record["source"] = str(result_path.relative_to(root))
            summaries.append(record)

for record in summaries:
    dataset = record["dataset_name"].rsplit("/", maxsplit=1)[-1]
    attack = f"{record['attack_type']}-{record['attack_position']}"
    prompt_mode = record.get("prompt_mode", "standard")
    print(
        f"{dataset} | {record['model_name']} | {prompt_mode} | {attack} | "
        f"{record['flipped_count']}/{record['total_queries']} | "
        f"{record['flipped_percentage']:.2f}%"
    )

trec-dl-2019 | openai.gpt-oss-20b-1:0 | standard | so-back | 4060/4060 | 100.00%
trec-dl-2020 | openai.gpt-oss-20b-1:0 | defense | so-back | 1433/3864 | 37.09%
trec-dl-2020 | openai.gpt-oss-20b-1:0 | standard | so-back | 4080/4080 | 100.00%
trec-dl-2020 | meta.llama3-70b-instruct-v1:0 | standard | so-back | 4006/4096 | 97.80%
trec-dl-2020 | meta.llama3-8b-instruct-v1:0 | defense | so-back | 2318/4096 | 56.59%
trec-dl-2020 | meta.llama3-8b-instruct-v1:0 | standard | so-back | 2462/4096 | 60.11%
trec-dl-2019 | qwen.qwen3-32b-v1:0 | defense | sd-back | 663/4096 | 16.19%
trec-dl-2019 | qwen.qwen3-32b-v1:0 | standard | sd-back | 3972/4096 | 96.97%
trec-dl-2019 | qwen.qwen3-32b-v1:0 | standard | so-back | 3928/4096 | 95.90%
trec-dl-2019 | qwen.qwen3-32b-v1:0 | defense | so-back | 1189/4096 | 29.03%
trec-dl-2020 | qwen.qwen3-32b-v1:0 | defense | so-back | 1157/4096 | 28.25%
trec-dl-2020 | qwen.qwen3-32b-v1:0 | standard | so-back | 3773/4096 | 92.11%


In [7]:
from collections import defaultdict, deque


detail_summaries = []

for detail_path in sorted(output_dir.rglob("detail_*pairwise*.json")):
    with detail_path.open(encoding="utf-8") as detail_file:
        details = json.load(detail_file)

    clean = [row for row in details if row["phase"] == "original"]
    attacked = [row for row in details if row["phase"] == "attacked"]
    clean_by_pair = defaultdict(deque)
    for row in clean:
        key = (row["query"], row["doc1_id"], row["doc2_id"])
        clean_by_pair[key].append(row)

    matched = []
    for after in attacked:
        key = (after["query"], after["doc1_id"], after["doc2_id"])
        if clean_by_pair[key]:
            matched.append((clean_by_pair[key].popleft(), after))
    flipped = [
        (before, after)
        for before, after in matched
        if before["label"] != after["label"]
    ]
    target_ids = {
        before["doc2_id"] if before["label"] == "A" else before["doc1_id"]
        for before, _ in flipped
    }
    query_texts = {before["query"] for before, _ in flipped}
    summary = {
        "source": str(detail_path.relative_to(root)),
        "flipped_pairs": len(flipped),
        "unique_attacked_passages": len(target_ids),
        "queries_with_flips": len(query_texts),
    }
    detail_summaries.append(summary)
    print(
        f"{detail_path.name} | flipped pairs={len(flipped)} | "
        f"unique attacked passages={len(target_ids)} | queries={len(query_texts)}"
    )

detail_gpt-oss-20b_pairwise_so_back_20260830_223417.json | flipped pairs=4060 | unique attacked passages=2147 | queries=36
detail_gpt-oss-20b_trec-dl-2020_pairwise_so_back_defense_20260831_200827.json | flipped pairs=1652 | unique attacked passages=1403 | queries=46
detail_gpt-oss-20b_trec-dl-2020_pairwise_so_back_standard_20260831_142001.json | flipped pairs=4081 | unique attacked passages=2626 | queries=46
detail_Llama3-70B_trec-dl-2020_pairwise_so_back_standard_20260831_202453.json | flipped pairs=4006 | unique attacked passages=2215 | queries=46
detail_Llama3-8B_trec-dl-2020_pairwise_so_back_defense_20260831_141242.json | flipped pairs=2318 | unique attacked passages=1926 | queries=46
detail_Llama3-8B_trec-dl-2020_pairwise_so_back_standard_20260831_131533.json | flipped pairs=2462 | unique attacked passages=2003 | queries=46
detail_Qwen3-32B_pairwise_sd_back_defense_20260831_121004.json | flipped pairs=663 | unique attacked passages=258 | queries=36
detail_Qwen3-32B_pairwise_sd_bac

## Marker-aware defense

Pairwise jobs accept `--prompt_mode defense`. The defense evaluator treats marker-associated instructions as untrusted passage content and ranks only substantive relevance. The same evaluator prompt is used for the clean projection and attacked comparison. Result summaries and detailed rows record `prompt_mode`, and the table reports standard and defended ASR separately.

In [8]:
defense_summaries = [
    record for record in summaries if record.get("prompt_mode") == "defense"
]
if not defense_summaries:
    print("No completed pairwise defense runs found.")
else:
    for record in defense_summaries:
        dataset = record["dataset_name"].rsplit("/", maxsplit=1)[-1]
        print(
            f"{dataset} | {record['model_name']} | "
            f"{record['attack_type']}-{record['attack_position']} | "
            f"{record['flipped_count']}/{record['total_queries']} | "
            f"{record['flipped_percentage']:.2f}%"
        )

trec-dl-2020 | openai.gpt-oss-20b-1:0 | so-back | 1433/3864 | 37.09%
trec-dl-2020 | meta.llama3-8b-instruct-v1:0 | so-back | 2318/4096 | 56.59%
trec-dl-2019 | qwen.qwen3-32b-v1:0 | sd-back | 663/4096 | 16.19%
trec-dl-2019 | qwen.qwen3-32b-v1:0 | so-back | 1189/4096 | 29.03%
trec-dl-2020 | qwen.qwen3-32b-v1:0 | so-back | 1157/4096 | 28.25%


## Interpretation

The paper-style ASR numerator is the number of flipped ordered pair instances. On TREC-DL 2019, Qwen3-32B defense reduced DOH from 95.90% to 29.03% (66.87 percentage points) and DCH from 96.97% to 16.19% (80.79 points). On TREC-DL 2020 DOH, Qwen3-32B standard reached 92.11% with 2,467 distinct successful target passages. Llama 3 8B defense reduced flips from 2,462 (60.11%) to 2,318 (56.59%): 144 fewer flips and a 3.52 percentage-point reduction; distinct successful target passages fell from 2,003 to 1,926. All 2020 runs had 4,096 valid clean and attacked outputs across 46 query texts. GPT-OSS-20B currently has only a 2019 standard DOH result: 4,060/4,060 valid pairs (100.00%). Its 36 invalid clean outputs are why detailed rows are aligned by query and document IDs rather than list position.